In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Preprocessing

### 1. Converting to lowercase

In [5]:
df["review"] = df["review"].str.lower()

### 2. Removing URL

In [6]:
import re

def remove_URL(data):
    text = re.sub(r"http\S+","",data)            # (Pattern , repl , string)
    return text

df["review"] = df["review"].apply(remove_URL)

### 3. Remove Punctuations

In [7]:
def remove_punctuations(data):
    text = re.sub(r"[^A-Za-z0-9\s]","",data)            # (Pattern , repl , string)
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML tags

In [8]:
def remove_html(data):
    text = re.sub(r"<.*?>","",data)            # (Pattern , repl , string)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing Stopwords

In [9]:
import nltk 

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [11]:
def remove_stopwords(text):
    stop_words = stopwords.words("english")
    tokens = word_tokenize(text)

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text

df["review"] = df["review"].apply(remove_stopwords)

### 6. Stemming 

In [12]:
from nltk.stem import PorterStemmer
# running ---> run

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [14]:
y = df["sentiment"]

### 8. Vectorization

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(max_features=5000)

x = vec.fit_transform(df["review"])

In [16]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2868)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4369)	0.041994187696759305
  (0, 4171)	0.17799685402440263
  (0, 3693)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3805)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4497)	0.07614066339174266
  (0, 3857)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

### Dataset & DataLoaders

In [17]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    x , y , test_size = 0.2 , random_state=42
)
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248304 stored elements and shape (39665, 5000)>

In [18]:
X_train = X_train.toarray()        # Convert sparse matrix to nparray 
X_test = X_test.toarray()

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset , DataLoader

In [20]:
trainset = TensorDataset(
    torch.from_numpy(X_train).float(),           # Converts nparray to tensor dataset
    torch.from_numpy(y_train.values).float()
)

testset = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [21]:
train_loader = DataLoader(trainset , shuffle=True , batch_size=64)
test_loader = DataLoader(testset , shuffle = True , batch_size=64)

## RNN Architechture 